# Pipeline de preprocesamiento reproducible — LaLiga

Este notebook documenta la procedencia, el contrato de columnas, la limpieza y la unión de los dos CSV originales. Los archivos de `data/raw/` se leen sin modificarlos; la única salida tabular se guarda en `data/processed/laliga_matches_clean.csv`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.laliga_loader import (
    SOURCE_PROVENANCE, audit_dataset, build_source_column_policy,
    file_sha256, load_processed_dataset, load_raw_sources,
    preprocess_sources,
)
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'laliga_matches_clean.csv'

## 1. Fuentes y huellas

- `LaLiga_Matches.csv`: consolidado histórico identificado en Kaggle; la ficha atribuye los archivos a sus autores originales y se registra Football-Data como fuente aguas arriba.
- `laliga_2025_2026_stats.csv`: instantánea del CSV `SP1.csv` de Football-Data para 2025/26.

La licencia/condición de reutilización requiere aprobación del equipo y por eso el gate `Data Ready` permanece abierto.

In [2]:
provenance = pd.DataFrame.from_dict(SOURCE_PROVENANCE, orient='index')
provenance.index.name = 'archivo_raw'
display(provenance[['source_name', 'publisher', 'source_page_url', 'acquisition', 'license', 'license_status']])
pd.DataFrame([
    {'archivo': path.name, 'bytes': path.stat().st_size, 'sha256': file_sha256(path)}
    for path in sorted(RAW_DIR.glob('*.csv'))
])

,source_name,publisher,source_page_url,acquisition,license,license_status
archivo_raw,,,,,,
LaLiga_Matches.csv,La Liga Complete Dataset,Kishan Kumar (Kaggle),https://www.kaggle.com/datasets/kishan305/la-l...,Descarga manual del CSV consolidado publicado ...,Data files © Original Authors (según la ficha ...,pending_team_approval
laliga_2025_2026_stats.csv,Football-Data Spain La Liga 2025/2026 (SP1.csv),Football-Data.co.uk,https://www.football-data.co.uk/data.php,Descarga del CSV SP1 de la temporada 2025/2026...,Football-Data permite acceso gratuito y declar...,pending_team_approval


,archivo,bytes,sha256
0,laliga_2025_2026_stats.csv,203623,22f754836d287254c9b4fc85001192a235f468a56bde5d...
1,LaLiga_Matches.csv,582442,d5d36d0ffff6dc73697e5bf794e19f0546fde07a77fe8c...


## 2. Perfil previo a la limpieza

Se comprueban dimensiones, duplicados exactos y nulos antes de transformar. Los nulos de descanso son opcionales; los campos críticos para identificar y etiquetar el partido son fecha, equipos, goles finales y resultado final.

In [3]:
historical, detailed = load_raw_sources(RAW_DIR)
pd.DataFrame([
    {'archivo': 'LaLiga_Matches.csv', 'filas': len(historical), 'columnas': historical.shape[1], 'duplicados_exactos': historical.duplicated().sum(), 'celdas_nulas': historical.isna().sum().sum()},
    {'archivo': 'laliga_2025_2026_stats.csv', 'filas': len(detailed), 'columnas': detailed.shape[1], 'duplicados_exactos': detailed.duplicated().sum(), 'celdas_nulas': detailed.isna().sum().sum()},
])

,archivo,filas,columnas,duplicados_exactos,celdas_nulas
0,LaLiga_Matches.csv,11664,10,0,6
1,laliga_2025_2026_stats.csv,380,131,0,4295


## 3. Política completa de columnas

El histórico conserva sus campos de identificación, marcador y resultado. En el archivo detallado se conservan 39 columnas: identidad/tiempo, marcador, estadísticas agregadas y promedios de mercado. Se descartan cuotas por casa, máximas y de exchange porque son redundantes respecto de los promedios, elevan la dimensionalidad y tienen cobertura irregular.

In [4]:
column_policy = build_source_column_policy(historical, detailed)
display(column_policy.groupby(['source_file', 'action']).size().rename('columnas').to_frame())
column_policy

columnas
source_file                action                        
LaLiga_Matches.csv         keep_and_rename              9
                           validate_then_derive         1
laliga_2025_2026_stats.csv drop                        92
                           keep_and_rename             39

,source_file,source_column,action,canonical_column,reason
0,LaLiga_Matches.csv,Season,validate_then_derive,season_validation_only,Se usa para comprobar la temporada original; l...
1,LaLiga_Matches.csv,Date,keep_and_rename,match_date,Campo mínimo necesario para identificar el par...
2,LaLiga_Matches.csv,HomeTeam,keep_and_rename,home_team,Campo mínimo necesario para identificar el par...
3,LaLiga_Matches.csv,AwayTeam,keep_and_rename,away_team,Campo mínimo necesario para identificar el par...
4,LaLiga_Matches.csv,FTHG,keep_and_rename,home_goals_ft,Campo mínimo necesario para identificar el par...
...,...,...,...,...,...
136,laliga_2025_2026_stats.csv,MaxCAHA,drop,,"Cuota específica, máxima o de exchange redunda..."
137,laliga_2025_2026_stats.csv,AvgCAHH,keep_and_rename,odds_avg_asian_home_close,Promedio de mercado seleccionado para evitar c...
138,laliga_2025_2026_stats.csv,AvgCAHA,keep_and_rename,odds_avg_asian_away_close,Promedio de mercado seleccionado para evitar c...
139,laliga_2025_2026_stats.csv,BFECAHH,drop,,"Cuota específica, máxima o de exchange redunda..."


## 4. Limpieza y criterio de combinación

Reglas: recortar texto, normalizar H/D/A, convertir fecha y goles, retirar duplicados exactos, retirar filas inequívocamente inválidas y deduplicar cada fuente por `fecha normalizada + local + visitante`. Después se hace una unión vertical. Cuando ambas fuentes contienen el mismo partido se conserva la fila detallada porque incluye el bloque mínimo del histórico y añade estadísticas/cuotas promedio.

In [5]:
clean, preprocessing = preprocess_sources(historical, detailed)
display(pd.DataFrame(preprocessing['source_cleaning']).T)
display(pd.Series(preprocessing['join'], name='valor').to_frame())
pd.Series(audit_dataset(clean), name='resultado').to_frame()

,input,exact_duplicate_rows_removed,invalid_rows_removed,invalid_breakdown_before_union,duplicate_match_keys_removed,season_mismatches_against_derived,rows_after_source_cleaning
LaLiga_Matches.csv,"{'filename': 'LaLiga_Matches.csv', 'rows': 116...",0,0,"{'invalid_date': 0, 'missing_critical': 0, 'bl...",0,437,11664
laliga_2025_2026_stats.csv,"{'filename': 'laliga_2025_2026_stats.csv', 'ro...",0,0,"{'invalid_date': 0, 'missing_critical': 0, 'bl...",0,0,380


,valor
type,vertical_union_with_overlap_precedence
key,"[normalized_match_date, trimmed_home_team, tri..."
why,Las fuentes describen la misma unidad (partido...
overlap_rows,100
historical_overlap_rows_discarded,100
detailed_rows_prioritized,100


,resultado
rows,11944
columns,54
season_count,31
date_min,1995-09-02
date_max,2026-05-24
duplicate_rows,0
duplicate_match_ids,0
missing_target,0
missing_half_time_rows,2
full_time_result_inconsistencies,0


## 5. Persistencia y verificación de la salida

Los nulos del bloque detallado en temporadas históricas se mantienen: representan ausencia estructural, no un dato perdido imputable. Los dos registros con descanso incompleto también se conservan porque esas variables son opcionales y postpartido.

In [6]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(PROCESSED_PATH, index=False, encoding='utf-8', date_format='%Y-%m-%d')
reloaded = load_processed_dataset(PROCESSED_PATH)
reloaded_audit = audit_dataset(reloaded)
pd.DataFrame({
    'control': ['ruta', 'filas', 'columnas', 'sha256', 'duplicados_id', 'target_nulo'],
    'valor': [str(PROCESSED_PATH.relative_to(PROJECT_ROOT)), len(reloaded), reloaded.shape[1], file_sha256(PROCESSED_PATH), reloaded_audit['duplicate_match_ids'], reloaded_audit['missing_target']],
})

,control,valor
0,ruta,data\processed\laliga_matches_clean.csv
1,filas,11944
2,columnas,54
3,sha256,6288a872df07a196a48ea05039671feba0616489927ebc...
4,duplicados_id,0
5,target_nulo,0


## Conclusión

La salida contiene una fila por partido y un esquema canónico único. La prioridad de la fuente detallada evita duplicar los encuentros compartidos y conserva la máxima información. No se imputan ausencias estructurales ni se eliminan partidos deportivos extremos que sean lógicamente válidos.